# Pydantic Notes

Pydantic is used for **data and type validation** in Python.

---

## Two Main Problems It Solves

---

### 1) Type Validation

#### Problem

Python is dynamically typed — variables can hold any value regardless of type. Consider a function that inserts data into a database:

```python
def insert_patient(name, age):
    print(name)
    print(age)
    print('inserted into database')

insert_patient('sameen', '30')  # age passed as string — bad!
```

Even with type hints, Python won't enforce them at runtime:

```python
def insert_patient(name: str, age: int):
    print(name)
    print(age)
    print('inserted into database')

insert_patient('sameen', '30')  # Still accepts a string for age!
```

Adding manual `if-else` type checks works but doesn't scale:

```python
def insert_patient(name: str, age: int):
    if type(name) == str and type(age) == int:
        if age < 0:
            raise ValueError("Age can't be negative")
        print(name)
        print(age)
        print('inserted into database')
    else:
        raise TypeError('Incorrect data type')

def update_patient(name: str, age: int):
    if type(name) == str and type(age) == int:
        if age < 0:
            raise ValueError("Age can't be negative")
        print(name)
        print(age)
        print('updated in database')
    else:
        raise TypeError('Incorrect data type')

# ... repeated for every function — not scalable!
```

> **Problem:** For `n` functions, you repeat the same boilerplate validation code. This is not scalable for production.

---

#### Solution — Pydantic Model (3 Steps)

**Step 1 — Define a Pydantic Model**

Create a class representing the ideal schema of the data:

```python
from pydantic import BaseModel

class Patient(BaseModel):
    name: str
    age: int
```

**Step 2 — Instantiate the Model with raw input data**

Pass your data dict into the class — Pydantic handles type validation automatically:

```python
patient_info = {'name': 'nitish', 'age': 30}
patient_object = Patient(**patient_info)  # **patient_info unpacks the dict as keyword args
```

**Step 3 — Pass the validated object to your functions**

```python
def insert_patient(patient: Patient):
    print(patient.name)
    print(patient.age)
    print('inserted into database')

patient_info = {'name': 'nitish', 'age': 30}
patient_object = Patient(**patient_info)
insert_patient(patient_object)
```

**Full Example:**

```python
from pydantic import BaseModel

class Patient(BaseModel):
    name: str
    age: int

def insert_patient(patient: Patient):
    print(patient.name)
    print(patient.age)
    print('inserted into database')

patient_info = {'name': 'nitish', 'age': 30}
patient_object = Patient(**patient_info)
insert_patient(patient_object)
```

---

### Complex Pydantic Model

Use `List` and `Dict` from `typing` for nested types — this enables **double validation** (e.g. ensuring every element inside a list is a string):

```python
from pydantic import BaseModel
from typing import List, Dict

class Patient(BaseModel):
    name: str
    age: int
    weight: float
    married: bool
    allergies: List[str]        # validates that each item in the list is a string
    contact_details: Dict[str, str]

def insert_patient(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    for item in patient.allergies:
        print(item)
    for key, value in patient.contact_details.items():
        print(f"{key}: {value}")
    print('inserted into database')

patient_info = {
    'name': 'nitish',
    'age': 30,
    'weight': 32.3,
    'married': True,
    'allergies': ['pollen', 'flu'],
    'contact_details': {'email': 'abc@gmail.com', 'phone': '9556466545'}
}
patient_object = Patient(**patient_info)
insert_patient(patient_object)
```

---

### Optional Fields

By default, all fields defined in the model are **required**. Use `Optional` to mark fields as not mandatory:

```python
from pydantic import BaseModel
from typing import List, Dict, Optional

class Patient(BaseModel):
    name: str
    age: int
    weight: float
    married: bool
    allergies: Optional[List[str]] = None   # not required; defaults to None
    contact_details: Dict[str, str]
```

---

## 2) Data Validation

There are three types of Pydantic data validation.

---

### a) Custom Datatypes & Field Constraints

Use built-in special types like `EmailStr` and `AnyUrl`, and use `Field` to set constraints or ranges:

```python
from pydantic import BaseModel, Field, EmailStr, AnyUrl
from typing import List, Dict, Optional

class Patient(BaseModel):
    name: str = Field(max_length=50)
    age: int
    weight: float = Field(gt=0)     # gt=0 means weight must be greater than 0
    email: EmailStr                  # validates email format
    url: AnyUrl                      # validates URL format
    married: bool
    allergies: Optional[List[str]] = None
    contact_details: Dict[str, str]
```

Use `Annotated` with `Field` to attach metadata to a field:

```python
from pydantic import BaseModel, Field, EmailStr, AnyUrl
from typing import List, Dict, Optional, Annotated

class Patient(BaseModel):
    name: str = Field(max_length=50)
    age: Annotated[int, Field(title="Age of the person", default=None)]  # Annotated[Datatype, Field()]
    weight: float = Field(gt=0)
    email: EmailStr
    url: AnyUrl
    married: bool
    allergies: Optional[List[str]] = None
    contact_details: Dict[str, str]
```

> Using `Field` you can set custom constraints, default values, and metadata.

---

### b) Field Validator

Use `@field_validator` to write custom validation logic for a specific field. For example, ensuring an email belongs to a valid bank domain:

```python
from pydantic import BaseModel, EmailStr, field_validator

class Patient(BaseModel):
    name: str
    age: int
    weight: float
    email: EmailStr
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]

    @field_validator('email')
    @classmethod
    def email_validator(cls, value):
        valid_domains = ['hdfc.com', 'icici.com']
        domain_name = value.split('@')[-1]
        if domain_name not in valid_domains:
            raise ValueError('Not a valid domain')
        return value

    @field_validator('name')
    @classmethod
    def transform_name(cls, value):
        return value.upper()
```

**Basic syntax:**

```python
@field_validator('field_name')
@classmethod
def function_name(cls, value):
    # put constraint here
    return value
```

**Validator modes:**

| Mode | Behaviour |
|------|-----------|
| `mode='before'` | Receives value **before** type coercion |
| `mode='after'`  | Receives value **after** type coercion (default) |

---

### c) Model Validator

Use `@model_validator` when validation logic spans **more than one field**. For example, if a patient is older than 60, their contact details must include an emergency number:

```python
from pydantic import BaseModel, model_validator

class Patient(BaseModel):
    name: str
    age: int
    weight: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]

    @model_validator(mode='after')
    @classmethod
    def validate_emergency_contact(cls, model):
        if model.age > 60 and 'emergency' not in model.contact_details:
            raise ValueError('Patients older than 60 must have an emergency contact')
        return model
```

---

## 3) Computed Fields

Derive a new field from existing ones using `@computed_field`. For example, computing BMI from height and weight:

```python
from pydantic import BaseModel, computed_field

class Patient(BaseModel):
    name: str
    age: int
    weight: float
    height: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]

    @computed_field
    @property
    def calculate_bmi(self) -> float:
        bmi = round(self.weight / (self.height ** 2), 2)
        return bmi

def insert_patient(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.calculate_bmi)
    print('inserted into database')

patient_info = {
    'name': 'nitish',
    'age': 30,
    'weight': 32.3,
    'height': 56.3,
    'married': True,
    'allergies': ['pollen', 'flu'],
    'contact_details': {'email': 'abc@gmail.com', 'phone': '9556466545'}
}
patient_object = Patient(**patient_info)
insert_patient(patient_object)
```

---

## 4) Nested Models

Use one Pydantic model as a field inside another model. This handles complex nested data structures cleanly:

```python
from pydantic import BaseModel

class Address(BaseModel):
    city: str
    pincode: int
    state: str

class Patient(BaseModel):
    name: str
    gender: str
    age: int
    address: Address       # Address model used as a nested field

address_dict = {'city': 'Pune', 'pincode': 411015, 'state': 'Maharashtra'}
address_object = Address(**address_dict)

patient_dict = {'name': 'Sameen', 'gender': 'Male', 'age': 34, 'address': address_object}
patient_object = Patient(**patient_dict)
```

> Since `address` can have many fields of varying types (`list`, `str`, `int`, etc.), wrapping it in its own model ensures proper validation across all nested fields.

---

## 5) Serialization

Convert a Pydantic model object into a dictionary or JSON. **Mostly used in FastAPI for building APIs.**

```python
# Convert to dictionary
patient_object.model_dump()

# Convert to JSON
patient_object.model_dump_json()
```

You can include or exclude specific fields during serialization:

```python
# Include only specific fields
patient_object.model_dump(include=['name', 'age'])

# Exclude specific fields
patient_object.model_dump(exclude=['allergies', 'contact_details'])
```